In [ ]:

ZIP_PATH = "/content/drive/MyDrive/state-farm-distracted-driver-detection.zip"
TARGET_DIR = "/content/statefarm"

os.makedirs(TARGET_DIR, exist_ok=True)

# 압축 풀기
!unzip -q "{ZIP_PATH}" -d "{TARGET_DIR}"

# 잘 풀렸는지 체크
!ls -R /content/statefarm | head -n 40

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
import os

# 1. Kaggle API 설정
!pip install -q kaggle
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("kaggle.json 파일을 업로드해주세요.")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# 2. 데이터셋 다운로드 및 압축 해제
TARGET_DIR = "/content/statefarm"
if not os.path.exists(TARGET_DIR):
    !kaggle competitions download -c state-farm-distracted-driver-detection
    !mkdir -p {TARGET_DIR}
    !unzip -q state-farm-distracted-driver-detection.zip -d {TARGET_DIR}
    print("✅ 데이터 다운로드 및 압축 해제 완료!")

In [ ]:
import os, random, numpy as np, tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)  # python/numpy/tf 한 번에
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print("determinism enable failed:", e)

import os, json
import numpy as np
import pandas as pd
from pathlib import Path

# ===== 설정 =====
R_TRAIN, R_VAL, R_TEST = 0.8, 0.1, 0.1
SPLIT_JSON = "/content/drive/MyDrive/statefarm_split_driver_8_1_1_seed42.json"

# BASE_DIR가 이미 있으면 그걸 쓰고, 없으면 /content/statefarm에서 자동 탐색
try:
    BASE_DIR
except NameError:
    BASE_DIR = "/content/statefarm"

# driver_imgs_list.csv 찾기
csv_files = list(Path(BASE_DIR).rglob("driver_imgs_list.csv"))
assert csv_files, f"driver_imgs_list.csv를 {BASE_DIR} 아래에서 못 찾았어."
CSV_PATH = str(csv_files[0])
ROOT = str(Path(CSV_PATH).parent)

# imgs/train 찾기
def find_train_dir(root):
    root = Path(root)
    for p in root.rglob("imgs"):
        if (p / "train").exists():
            return str(p / "train")
    raise FileNotFoundError("imgs/train 폴더를 못 찾았어.")
TRAIN_DIR = find_train_dir(ROOT)

print("CSV_PATH:", CSV_PATH)
print("TRAIN_DIR:", TRAIN_DIR)

drivers_df = pd.read_csv(CSV_PATH)  # columns: subject, classname, img
drivers_df["label"] = drivers_df["classname"].str.replace("c", "", regex=False).astype(int)
drivers_df["path"]  = drivers_df.apply(lambda r: os.path.join(TRAIN_DIR, r["classname"], r["img"]), axis=1)

def make_or_load_driver_split(df, seed, r_train, r_val, save_path):
    subjects_all = np.array(sorted(df["subject"].unique()))  # 항상 같은 시작 순서

    if os.path.exists(save_path):
        with open(save_path, "r") as f:
            obj = json.load(f)
        tr = np.array(obj["train_subjects"])
        va = np.array(obj["val_subjects"])
        te = np.array(obj["test_subjects"])
        # 데이터가 동일하면 그대로 사용
        if set(tr) | set(va) | set(te) == set(subjects_all):
            return tr, va, te
        print("⚠️ 저장된 split이 현재 driver 목록과 달라서 새로 생성할게.")

    rng = np.random.RandomState(seed)
    subjects = subjects_all.copy()
    rng.shuffle(subjects)

    n = len(subjects)
    n_train = int(n * r_train)
    n_val   = int(n * r_val)
    # 나머지는 test(holdout)
    train_sub = subjects[:n_train]
    val_sub   = subjects[n_train:n_train + n_val]
    test_sub  = subjects[n_train + n_val:]

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w") as f:
        json.dump({
            "seed": seed,
            "ratios": [r_train, r_val, 1.0 - r_train - r_val],
            "train_subjects": train_sub.tolist(),
            "val_subjects": val_sub.tolist(),
            "test_subjects": test_sub.tolist(),
        }, f, indent=2)

    return train_sub, val_sub, test_sub



train_subjects, val_subjects, test_subjects = make_or_load_driver_split(
    drivers_df, SEED, R_TRAIN, R_VAL, SPLIT_JSON
)

train_df = drivers_df[drivers_df["subject"].isin(train_subjects)].reset_index(drop=True)
val_df   = drivers_df[drivers_df["subject"].isin(val_subjects)].reset_index(drop=True)
test_df  = drivers_df[drivers_df["subject"].isin(test_subjects)].reset_index(drop=True)

print("driver counts | train:", len(train_subjects), "val:", len(val_subjects), "test:", len(test_subjects))
print("image  counts | train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
print("Saved split:", SPLIT_JSON)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def show_per_class_samples(df, k=1, cols=5, title="Per-class samples", seed=42, resize_to=None):
    """
    df: columns must include ['path', 'label'] and/or ['classname']
    k: 클래스당 몇 장 뽑을지
    cols: 출력 컬럼 수 (10클래스면 5로 두면 2줄로 보기 좋음)
    resize_to: (W,H) or None
    """
    rng = np.random.RandomState(seed)

    # label 준비 (없으면 classname에서 생성)
    if "label" not in df.columns:
        assert "classname" in df.columns, "df에 label도 classname도 없어."
        tmp = df.copy()
        tmp["label"] = tmp["classname"].str.replace("c", "", regex=False).astype(int)
        df2 = tmp
    else:
        df2 = df

    # 클래스 목록 정렬(0~9)
    classes = sorted(df2["label"].unique().tolist())
    # 보통 0~9라 10개
    total = len(classes) * k
    rows = int(np.ceil(total / cols))

    plt.figure(figsize=(cols * 4, rows * 4))
    plt.suptitle(title, fontsize=14)

    plot_i = 1
    for c in classes:
        sub = df2[df2["label"] == c]
        if len(sub) == 0:
            continue

        # 해당 클래스에서 k개 샘플
        idxs = rng.choice(len(sub), size=min(k, len(sub)), replace=False)
        sub = sub.iloc[idxs]

        for _, row in sub.iterrows():
            path = row["path"]
            img = Image.open(path).convert("RGB")
            if resize_to is not None:
                img = img.resize(resize_to)

            plt.subplot(rows, cols, plot_i)
            plt.imshow(img)
            plt.axis("off")

            # title 표시
            cls_name = f"c{int(c)}"
            extra = []
            if "subject" in row.index:
                extra.append(f"drv={row['subject']}")
            plt.title(cls_name + (" | " + " ".join(extra) if extra else ""), fontsize=10)

            plot_i += 1

    plt.tight_layout()
    plt.show()

# 사용 예시:
show_per_class_samples(train_df, k=1, cols=5, title="TRAIN: 1 sample per class", resize_to=(384,384))


In [ ]:
# =========================
# train_paths / val_paths 생성
# =========================
import numpy as np
import tensorflow as tf

assert "train_df" in globals() and "val_df" in globals(), "공통 split 셀(8:1:1)을 먼저 실행해서 train_df/val_df를 만들어야 해."

train_paths  = train_df["path"].astype(str).to_list()
train_labels = train_df["label"].astype(np.int32).to_list()

val_paths    = val_df["path"].astype(str).to_list()
val_labels   = val_df["label"].astype(np.int32).to_list()

print("train_paths example:", train_paths[0])
print("train_labels example:", train_labels[0])
print("sizes:", len(train_paths), len(val_paths))

# SEED 없으면 기본값
try:
    SEED
except NameError:
    SEED = 42
tf.keras.utils.set_random_seed(SEED)

# =========================
# 설정
# =========================
IMG_SIZE   = 320
BATCH_SIZE = 32
EPOCHS     = 15

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)

# =========================
# ROI: ratio slice -> resize (head/hand/left_low)
# =========================
def _safe_slice_resize(img, y0, y1, x0, x1, out_size):
    H = tf.shape(img)[0]
    W = tf.shape(img)[1]

    y0 = tf.clip_by_value(y0, 0, H-1)
    y1 = tf.clip_by_value(y1, y0+1, H)
    x0 = tf.clip_by_value(x0, 0, W-1)
    x1 = tf.clip_by_value(x1, x0+1, W)

    crop = img[y0:y1, x0:x1, :]
    crop = tf.image.resize(crop, [out_size, out_size])
    return crop

def make_three_views(img, out_size, training=False):
    """
    return: full, head, hand, left_low
    val/test에는 증강 없음. train에만 약한 색상 증강 적용.
    """
    H = tf.shape(img)[0]
    W = tf.shape(img)[1]

    # full
    full = tf.image.resize(img, [out_size, out_size])

    # head ROI (네가 준 값)
    y0_head = tf.cast(0.00 * tf.cast(H, tf.float32), tf.int32)
    y1_head = tf.cast(0.50 * tf.cast(H, tf.float32), tf.int32)
    x0_head = tf.cast(0.05 * tf.cast(W, tf.float32), tf.int32)
    x1_head = tf.cast(0.60 * tf.cast(W, tf.float32), tf.int32)
    head = _safe_slice_resize(img, y0_head, y1_head, x0_head, x1_head, out_size)

    # hand ROI (네가 준 값, right-bottom)
    y0_hand = tf.cast(0.15 * tf.cast(H, tf.float32), tf.int32)
    y1_hand = H
    x0_hand = tf.cast(0.55 * tf.cast(W, tf.float32), tf.int32)
    x1_hand = W
    hand = _safe_slice_resize(img, y0_hand, y1_hand, x0_hand, x1_hand, out_size)

    # train에만 약한 색상 증강 (val에는 절대 X)
    if training:
        def aug(x):
            x = tf.image.random_brightness(x, 0.12)
            x = tf.image.random_contrast(x, 0.9, 1.1)
            return x
        full = aug(full); head = aug(head); hand = aug(hand);
    return full, head, hand

# =========================
# tf.data (4-input)  ✅ dict 반환 (안전)
# =========================
AUTOTUNE = tf.data.AUTOTUNE

def make_ds_resnet_triple(paths, labels, training: bool):
    paths  = tf.constant(paths)
    labels = tf.constant(labels, dtype=tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(min(len(paths), 4096), seed=SEED, reshuffle_each_iteration=False)

    options = tf.data.Options()
    options.experimental_deterministic = True
    ds = ds.with_options(options)

    def _load(path, label):
        img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        img = tf.cast(img, tf.float32)  # 0..255

        x_full, x_head, x_hand = make_three_views(img, IMG_SIZE, training)

        # preprocess_input (각 뷰에 동일 적용)
        x_full    = tf.keras.applications.resnet.preprocess_input(x_full)
        x_head    = tf.keras.applications.resnet.preprocess_input(x_head)
        x_hand    = tf.keras.applications.resnet.preprocess_input(x_hand)

        x = {
            "full": x_full,
            "head": x_head,
            "hand": x_hand,
        }
        return x, label

    ds = ds.map(_load, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds_resnet_triple(train_paths, train_labels, training=True)
val_ds   = make_ds_resnet_triple(val_paths,   val_labels,   training=False)

# =========================
# 헤드 (네가 쓰고 싶다던 PreprocCompareHead 활용: branch feature용)
# =========================
class PreprocCompareHead(tf.keras.layers.Layer):
    def __init__(self, num_classes=10, p_drop=0.5, return_feat=False, **kwargs):
        super().__init__(**kwargs)
        self.gap  = tf.keras.layers.GlobalAveragePooling2D()
        self.drop = tf.keras.layers.Dropout(p_drop)
        self.fc   = tf.keras.layers.Dense(num_classes, dtype="float32")
        self.return_feat = return_feat

    def call(self, x, training=False):
        x = self.gap(x)
        x = self.drop(x, training=training)
        if self.return_feat:
            return x
        return self.fc(x)

def freeze_batch_norm(model):
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

# =========================
# 모델 (레즈넷 shared backbone, 4 inputs)
# =========================
class SEBlock(tf.keras.layers.Layer):
    def __init__(self, se_ratio=0.25, activation="swish", **kwargs):
        super().__init__(**kwargs)
        self.se_ratio = se_ratio
        self.activation = activation

    def build(self, input_shape):
        ch = int(input_shape[-1])
        r = max(1, int(ch * self.se_ratio))
        self.gap = tf.keras.layers.GlobalAveragePooling2D()
        self.fc1 = tf.keras.layers.Dense(r, activation=self.activation)
        self.fc2 = tf.keras.layers.Dense(ch, activation="sigmoid")
        self.reshape = tf.keras.layers.Reshape((1, 1, ch))
        self.mul = tf.keras.layers.Multiply()
        super().build(input_shape)

    def call(self, x, training=False):
        s = self.gap(x)          # (B, C)
        s = self.fc1(s)          # (B, C/r)
        s = self.fc2(s)          # (B, C)
        s = self.reshape(s)      # (B, 1, 1, C)
        return self.mul([x, s])  # channel-wise scale


inp_full    = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="full")
inp_head    = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="head")
inp_hand    = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="hand")

backbone = tf.keras.applications.ResNet101(
    include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling=None
)

for layer in backbone.layers:
    layer.trainable = False
for layer in backbone.layers:
    if "conv5_" in layer.name:   # 마지막 stage
        layer.trainable = True


freeze_batch_norm(backbone)

f_full = backbone(inp_full)
f_head = backbone(inp_head)
f_hand = backbone(inp_hand)

f_full = SEBlock(se_ratio=0.25, name="se_full")(f_full)
f_head = SEBlock(se_ratio=0.25, name="se_head")(f_head)
f_hand = SEBlock(se_ratio=0.25, name="se_hand")(f_hand)


branch_feat = PreprocCompareHead(num_classes=10, p_drop=0.5, return_feat=True, name="branch_feat")

v_full    = branch_feat(f_full)
v_head    = branch_feat(f_head)
v_hand    = branch_feat(f_hand)

feat = tf.keras.layers.Concatenate()([v_full, v_head, v_hand])
logits = tf.keras.layers.Dense(10, dtype="float32")(feat)

model = tf.keras.Model(
    inputs={"full": inp_full, "head": inp_head, "hand": inp_hand},
    outputs=logits,
    name="resnet101_roi3_preproc_compare"
)

# =========================
# 컴파일 + 체크포인트 + 학습
# =========================
try:
    opt = tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=3e-4)
except Exception:
    opt = tf.keras.optimizers.Adam(learning_rate=1e-4)

model.compile(
    optimizer=opt,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="acc")]
)




In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

def show_roi3_from_path(path, img_size=320):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.cast(img, tf.float32)  # 0..255

    full, head, hand= make_three_views(img, img_size, training=False)

    views  = [full, head, hand]
    titles = ["full", "head", "hand"]

    plt.figure(figsize=(16, 4))
    for i, (v, t) in enumerate(zip(views, titles), 1):
        plt.subplot(1, 4, i)
        plt.imshow(tf.cast(tf.clip_by_value(v, 0, 255), tf.uint8).numpy())
        plt.title(t)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# 사용 예시
show_roi3_from_path(train_paths[0], img_size=IMG_SIZE)
# show_roi4_from_path(val_paths[0], img_size=IMG_SIZE)


In [ ]:
ckpt_path = "/content/statefarm/best_resnet101_roi3.weights.h5"
ckpt = tf.keras.callbacks.ModelCheckpoint(
    ckpt_path,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[ckpt, early_stop],
    verbose=1
)

print("✅ best weights saved to:", ckpt_path)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 클래스 라벨(너가 쓰던 그대로)
class_labels = ['safe', 'text-R', 'talk-R', 'text-L', 'talk-L', 'radio', 'drink', 'reach', 'makeup', 'passenger']
n_classes = len(class_labels)

# =========================
# 1) 검증셋 전체 예측 (logits -> softmax)
# =========================
# model 출력이 logits이므로 softmax로 확률로 바꿔두면 분석할 때 편함
val_logits = model.predict(val_ds, verbose=0)  # shape (N, 10)
val_probs  = tf.nn.softmax(val_logits, axis=-1).numpy()
y_pred = np.argmax(val_probs, axis=1)

# y_true는 val_ds에서 순서대로 모으기
y_true = np.concatenate([y.numpy() for _, y in val_ds], axis=0)

# 안전 체크: val_paths와 길이 맞는지
print("N(val):", len(y_true), "| len(val_paths):", len(val_paths))
assert len(y_true) == len(val_paths), "val_paths와 val_ds의 샘플 수가 달라요. (split/paths 확인 필요)"

# =========================
# 2) Confusion Matrix
# =========================
cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))

plt.figure(figsize=(11, 9))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_labels, yticklabels=class_labels
)
plt.title('ResNet101 (3-Input) Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

print(classification_report(
    y_true, y_pred,
    labels=np.arange(n_classes),
    target_names=class_labels,
    digits=4
))

# =========================
# 3) 오답 시각화
# =========================
import cv2

def plot_misclassified_samples(actual_idx, pred_indices, title, num_images=5, show_prob=True):
    actual_idx = int(actual_idx)
    pred_indices = [int(p) for p in pred_indices]

    # 조건에 맞는 오답 인덱스 찾기
    mask = (y_true == actual_idx) & np.isin(y_pred, pred_indices)
    error_indices = np.where(mask)[0]

    if len(error_indices) == 0:
        print(f"\n[{title}] 해당되는 오답 사례가 없습니다.")
        return

    plt.figure(figsize=(4 * num_images, 5))
    for i, idx in enumerate(error_indices[:num_images]):
        img_path = val_paths[idx]
        img = cv2.imread(img_path)
        if img is None:
            print("이미지 로드 실패:", img_path)
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        t = int(y_true[idx])
        p = int(y_pred[idx])
        p_prob = float(val_probs[idx, p])
        t_prob = float(val_probs[idx, t])

        plt.subplot(1, num_images, i + 1)
        plt.imshow(img)

        if show_prob:
            plt.title(
                f"True: {class_labels[t]} (p={t_prob:.2f})\n"
                f"Pred: {class_labels[p]} (p={p_prob:.2f})",
                color='red', fontsize=11
            )
        else:
            plt.title(
                f"True: {class_labels[t]}\nPred: {class_labels[p]}",
                color='red', fontsize=11
            )

        plt.axis('off')

    plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# --- 1) Safe(0) -> Passenger(9) 오판 ---
plot_misclassified_samples(
    actual_idx=0,
    pred_indices=[9],
    title="[Case 1] Why 'Safe' was seen as 'Passenger'?",
    num_images=5
)

# --- 2) Passenger(9) -> Safe/Radio/Makeup 오판 ---
plot_misclassified_samples(
    actual_idx=9,
    pred_indices=[0, 5, 8],
    title="[Case 2] Why 'Passenger' was seen as Safe/Radio/Makeup?",
    num_images=5
)


In [ ]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np

BASE_DIR = "/content/statefarm"
TEST_DIR = os.path.join(BASE_DIR, "imgs", "test")
SAMPLE_SUB = os.path.join(BASE_DIR, "sample_submission.csv")

def make_submission_csv_roi3(
    model,
    ckpt_path,                 # best weights 경로
    img_size=320,
    batch_size=32,
    out_path="/content/submission_roi3.csv"
):
    assert os.path.exists(ckpt_path), f"weights 파일 없음: {ckpt_path}"
    model.load_weights(ckpt_path)
    print("✅ loaded:", ckpt_path)

    sub_df = pd.read_csv(SAMPLE_SUB)
    sub_df["path"] = sub_df["img"].apply(lambda x: os.path.join(TEST_DIR, x))

    # sanity
    print("example test path:", sub_df["path"].iloc[0])
    print("exists?:", os.path.exists(sub_df["path"].iloc[0]))
    test_paths = sub_df["path"].astype(str).tolist()
    print("n_test:", len(test_paths))

    AUTOTUNE = tf.data.AUTOTUNE

    def _load(path):
        img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        img = tf.cast(img, tf.float32)

        # ✅ test는 training=False로 고정(증강 없음)
        full, head, hand = make_three_views(img, img_size, training=False)

        # preprocess_input
        full     = tf.keras.applications.resnet.preprocess_input(full)
        head     = tf.keras.applications.resnet.preprocess_input(head)
        hand     = tf.keras.applications.resnet.preprocess_input(hand)

        # ✅ 모델 입력 dict 키와 반드시 동일해야 함: "full","head","hand","left_low"
        return {"full": full, "head": head, "hand": hand}

    ds = tf.data.Dataset.from_tensor_slices(test_paths)
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)

    logits = model.predict(ds, verbose=1)
    probs = tf.nn.softmax(logits, axis=1).numpy()

    out_df = pd.DataFrame(probs, columns=[f"c{i}" for i in range(probs.shape[1])])
    out_df.insert(0, "img", sub_df["img"].values)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    out_df.to_csv(out_path, index=False)
    print("✅ saved:", out_path)
    print(out_df.head())
    return out_df

# 사용 예시:
# best weights 경로는 학습 셀에서 저장한 ckpt_path를 그대로 넣으면 됨
# 예: ckpt_path = "/content/statefarm/best_effnetb4_roi4.weights.h5"
sub_csv_path = "/content/statefarm/submission_resnet101_roi3.csv"
make_submission_csv_roi3(
    model=model,
    ckpt_path="/content/statefarm/best_resnet101_roi3.weights.h5",
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    out_path=sub_csv_path
)
